# Lección 8.7 — Sistemas de recomendación: "gente como tú también disfrutó…"

> **Módulo 8 · Aprendizaje no supervisado** · Nivel 3 — Practicante
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 5.3 (KNN y distancias), Lección 8.4 (PCA), Lección 4.1 (RMSE)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar las grandes familias de recomendadores: **popularidad**, **basados en contenido**, **filtrado colaborativo** y **factorización de matrices**.
- Calcular a mano (y con código) la **similitud coseno** entre usuarios o productos.
- Entender la intuición de los **factores latentes**: "gustos ocultos" que el modelo descubre solo.
- Reconocer los problemas típicos: **arranque en frío**, **sesgo de popularidad** y la trampa de tratar "no visto" como "no me gustó".

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD

plt.rcParams["figure.figsize"] = (8, 4.5)
pd.set_option("display.precision", 2)
print("✅ Todo listo")

---
## 1. 📖 La intuición: el dependiente del videoclub 📼

Hace años, en los videoclubs había un dependiente que te conocía. Le decías: *"Me encantaron Rápido y Furioso y Duro de Matar"*, y él pensaba: *"Otros clientes a los que les gustaron esas dos también rentaron Misión Imposible… se la recomiendo."*

Ese dependiente hacía, sin saberlo, **filtrado colaborativo**: usaba los gustos de **gente parecida a ti** para adivinar qué te gustaría. No necesitaba saber de qué trataba la película; solo **quién vio qué**.

Netflix, Spotify, Amazon o YouTube hacen lo mismo con millones de personas. Y es **no supervisado** en su esencia: nadie etiqueta "esta película es para Ana"; el sistema descubre **patrones de gusto** en los datos.

### ✋ Ejercicio 1 — Calentamiento
Una amiga te recomienda un libro. ¿En qué dos situaciones distintas le harías caso, y cuál se parece más al dependiente del videoclub?
- a) "Te lo recomiendo porque es de ciencia ficción, como los que te gustan."
- b) "Te lo recomiendo porque a mí me encantaron los mismos 5 libros que a ti, y este también me encantó."

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Las dos son razonables, y son las dos grandes familias de recomendadores:
- **a)** es un recomendador **basado en contenido**: mira las **características** del producto (género, autor, precio) y busca parecidos a lo que ya te gustó.
- **b)** es **filtrado colaborativo**, como el dependiente: mira a **personas con gustos parecidos** a los tuyos. Ni siquiera necesita saber de qué trata el libro.
</details>

---
## 2. 📖 El concepto: la matriz de calificaciones

Todo empieza con una tabla: **usuarios en las filas, productos en las columnas** y, en cada celda, la calificación (de 1 a 5 estrellas). Casi todas las celdas están **vacías**: nadie ha visto todas las películas.

In [ ]:
peliculas = ["Rápido", "Duro", "Misión", "Titanic", "Notebook", "Amélie"]
calif = pd.DataFrame([
    [5,      4,      np.nan, 1,      np.nan, 2],      # Ana
    [4,      5,      5,      np.nan, 1,      np.nan], # Beto
    [5,      np.nan, 4,      2,      1,      1],      # Carla
    [1,      np.nan, 2,      5,      4,      np.nan], # Diego
    [np.nan, 1,      1,      4,      5,      5],      # Elena
    [2,      2,      np.nan, 5,      np.nan, 4],      # Fer
], index=["Ana", "Beto", "Carla", "Diego", "Elena", "Fer"], columns=peliculas)
calif

| Familia | Idea | Necesita | Ejemplo |
|---|---|---|---|
| **Popularidad** | Recomendar lo que más gusta a todos | Solo las calificaciones | "Lo más visto esta semana" |
| **Basado en contenido** | Productos parecidos a los que te gustaron | Características de los productos | "Porque viste películas de acción…" |
| **Filtrado colaborativo** (vecinos) | Usuarios parecidos a ti, o productos que gustan a la misma gente | Solo la matriz de calificaciones | "Quienes vieron esto también vieron…" |
| **Factorización de matrices** | Descubrir "gustos ocultos" que explican la matriz | Solo la matriz de calificaciones | El corazón del premio Netflix (2009) |
| **Híbridos** | Combinar todo lo anterior (y más) | Todo | Lo que usan las grandes plataformas |

### 🔮 Ejercicio 2
Mirando la tabla con sentido común: ¿qué calificación crees que le daría **Ana** a **Misión**? ¿Y a **Notebook**? ¿En qué te basaste?

✍️ *Mis predicciones:* Misión → ___ · Notebook → ___

<details><summary><b>👀 Ver respuesta</b></summary>

Ana ama la acción (Rápido 5, Duro 4) y no le gustan los dramas románticos (Titanic 1, Amélie 2). Beto y Carla tienen gustos parecidos y le dieron 5 y 4 a **Misión**: probablemente Ana le dé **4–5**. A **Notebook**, sus "gemelos" le dieron 1: probablemente **1–2**.

Acabas de hacer **filtrado colaborativo basado en usuarios** a mano. Ahora hagámoslo con números.
</details>

---
## 3. 📖 Cómo funciona: medir "gustos parecidos" con la similitud coseno

Para encontrar a los "gemelos de gusto" de Ana necesitamos medir qué tan parecidas son dos filas. Se usa la **similitud coseno**:
- Piensa en cada usuario como una **flecha** que apunta hacia sus gustos.
- Si dos flechas apuntan en la **misma dirección** → similitud **+1** (gustos idénticos).
- Si son perpendiculares → **0** (nada que ver).
- Si apuntan en direcciones **opuestas** → **−1** (lo que a uno le encanta, el otro lo odia).

Un truco importante antes de compararlos: **centrar** las calificaciones de cada usuario (restar su promedio). Así, un usuario "generoso" que da 4 a todo y uno "exigente" que da 2 a todo pueden tener los mismos gustos. Después, una celda vacía vale **0**, que significa "neutral, no sé".

In [ ]:
centradas = calif.sub(calif.mean(axis=1), axis=0)        # restamos el promedio de cada usuario
matriz = centradas.fillna(0).to_numpy()                   # vacío = 0 = "neutral"

def similitud_coseno(M):
    normas = np.linalg.norm(M, axis=1, keepdims=True)
    unit = M / normas
    return unit @ unit.T

sim_usuarios = pd.DataFrame(similitud_coseno(matriz), index=calif.index, columns=calif.index)
sim_usuarios.round(2)

### ✋ Ejercicio 3
1. ¿Quiénes son los dos usuarios **más parecidos** a Ana?
2. ¿Quién es su **opuesto**? ¿Tiene sentido mirando la tabla de calificaciones?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Carla** (≈0.66) y, bastante más lejos, **Beto** (≈0.17): los fans de la acción. Beto sale menos parecido porque solo comparte con Ana **dos** películas calificadas (Rápido y Duro); sus opiniones más fuertes (Misión y Notebook) caen justo donde Ana tiene huecos, que valen 0, y eso "diluye" la similitud. Con pocas películas en común, la similitud es poco confiable (lo verás en la tabla de problemas).
2. **Fer** (≈−0.97) es su opuesta casi perfecta, seguida de Diego (≈−0.80). A Fer le encantan Titanic y Amélie y le aburre la acción: justo al revés que Ana.

Ojo: un opuesto también es información útil. Si Fer odia algo, ¡quizá a Ana le guste!
</details>

### 📖 Predecir una calificación
Para estimar cuánto le gustaría a Ana una película que **no ha visto**:
1. Toma a los usuarios **parecidos** a Ana que sí la vieron.
2. Promedia lo que **se desviaron de su propio promedio** al calificarla, dando más peso a los más parecidos.
3. Súmale ese número al promedio de Ana.

In [ ]:
def predecir(usuario, pelicula, k=2):
    vieron = calif[pelicula].dropna().index.drop(usuario, errors="ignore")       # quién vio la película
    vecinos = sim_usuarios.loc[usuario, vieron].sort_values(ascending=False).head(k)   # los k más parecidos
    vecinos = vecinos[vecinos > 0]                                                # solo los que se parecen
    if vecinos.empty:
        return calif.loc[usuario].mean()                                          # sin vecinos: su promedio
    desvio = centradas.loc[vecinos.index, pelicula]
    return calif.loc[usuario].mean() + (vecinos * desvio).sum() / vecinos.sum()

for peli in ["Misión", "Notebook"]:
    print(f"Ana → {peli:9}: {predecir('Ana', peli):.1f} estrellas")

<details><summary><b>👀 ¿Coincidió con tu intuición del Ejercicio 2?</b></summary>

Misión ≈ **4.4** y Notebook ≈ **1.2**. Casi exactamente lo que razonaste a ojo: los vecinos de Ana (Carla y Beto) adoraron Misión y odiaron Notebook, así que el sistema "presta" esa opinión a Ana.
</details>

### ✋ Ejercicio 4 — Ahora con productos: "quienes vieron esto también vieron…"
En vez de comparar **usuarios** (filas), podemos comparar **películas** (columnas): dos películas son parecidas si **la misma gente** las califica de forma parecida. Es el filtrado colaborativo **basado en productos** (Amazon lo popularizó).

🔮 Antes de ejecutar: ¿qué película será la más parecida a **Titanic**?

✍️ *Mi apuesta:*

In [ ]:
sim_peliculas = pd.DataFrame(similitud_coseno(matriz.T), index=peliculas, columns=peliculas)
print("Más parecidas a Titanic:")
print(sim_peliculas["Titanic"].drop("Titanic").sort_values(ascending=False).round(2))

<details><summary><b>👀 Ver respuesta</b></summary>

Las más parecidas a Titanic son **Amélie** y **Notebook** (los dramas/románticas), y las menos parecidas son las de acción. **El sistema nunca supo el género de ninguna película**: lo descubrió solo porque la misma gente las califica igual.

Ventaja práctica del enfoque por productos: las similitudes entre productos cambian poco y se pueden **precalcular** (hay muchos menos productos que usuarios). Por eso es tan común en tiendas en línea.
</details>

---
## 4. 📖 Factorización de matrices: descubrir los "gustos ocultos"

Con 6 usuarios todo es fácil. Con millones, comparar a todos con todos es lento y la matriz está **vacía en un 99 %**. La idea ganadora del premio Netflix fue otra:

> Cada **usuario** se puede describir con unos pocos números ocultos ("cuánto le gusta la acción", "cuánto le gusta el romance"…), y cada **película** con los mismos números ("cuánta acción tiene", "cuánto romance"…). La calificación es, aproximadamente, **la combinación de ambos**.

```
            gustos ocultos                       calificación ≈ gusto · característica
Ana   = [acción: 2.0,  romance: -1.5]
Titanic = [acción: -0.5, romance: 2.0]     →  2.0·(-0.5) + (-1.5)·2.0 = -4  →  ¡no le gustará!
```

Lo mágico: **nadie le dice al modelo qué son esos números**. Los descubre solo, igual que PCA descubrió "tamaño corporal" en la Lección 8.4. De hecho, la herramienta que usaremos (`TruncatedSVD`) es pariente directa de PCA y funciona bien con matrices llenas de huecos.

Vamos a un caso más grande: **300 usuarios** y **40 películas** de 3 géneros. Cada usuario calificó solo ~25 % de las películas.

In [ ]:
rng = np.random.default_rng(0)
generos = ["Acción", "Romance", "Animada"]
nombres = [f"{generos[i % 3]}-{i // 3 + 1:02d}" for i in range(40)]     # Acción-01, Romance-01, Animada-01, ...
genero_peli = np.arange(40) % 3
calidad = rng.normal(0, 0.5, 40)                                      # algunas películas son simplemente mejores
gusto = rng.normal(0, 1.2, (300, 3))                                  # gusto oculto de cada usuario por cada género

real = 3.2 + gusto[:, genero_peli] + calidad + rng.normal(0, 0.5, (300, 40))
real = np.clip(np.round(real), 1, 5)
vista = rng.random((300, 40)) < 0.25                                  # solo vemos ~25 % de las celdas
grande = pd.DataFrame(np.where(vista, real, np.nan), columns=nombres)
print(f"Celdas con calificación: {vista.mean():.0%}")
grande.iloc[:6, :9]

Para evaluar honestamente, escondemos un 20 % de las calificaciones conocidas (**prueba**) y entrenamos con el resto, como en la Lección 2.5.

In [ ]:
filas, cols = np.where(vista)
es_prueba = rng.random(len(filas)) < 0.2
M = grande.to_numpy().copy()
M[filas[es_prueba], cols[es_prueba]] = np.nan                          # escondemos las de prueba
entren = pd.DataFrame(M, columns=nombres)
reales_prueba = grande.values[filas[es_prueba], cols[es_prueba]]

def rmse(pred):
    return np.sqrt(np.mean((pred - reales_prueba) ** 2))

# Líneas base (Lección 4.7)
promedio_global = np.nanmean(entren.values)
promedio_peli = entren.mean().fillna(promedio_global).values
print(f"Calificaciones de prueba: {es_prueba.sum()}")
print(f"RMSE 'siempre el promedio global'     : {rmse(np.full(es_prueba.sum(), promedio_global)):.2f}")
print(f"RMSE 'promedio de la película' (popul.): {rmse(promedio_peli[cols[es_prueba]]):.2f}")

### 🔮 Ejercicio 5
Ahora la factorización con `TruncatedSVD`. Le pedimos que descubra **3** gustos ocultos (sabemos que hay 3 géneros). ¿Crees que bajará el RMSE respecto a la popularidad? ¿Mucho o poco?

✍️ *Mi apuesta:*

In [ ]:
def factorizar(entren, n_factores):
    medias_usuario = entren.mean(axis=1)
    centr = entren.sub(medias_usuario, axis=0).fillna(0)                 # vacío = 0 = "neutral"
    svd = TruncatedSVD(n_components=n_factores, random_state=0)
    usuarios_f = svd.fit_transform(centr)                                # gustos ocultos de cada usuario
    reconstruida = usuarios_f @ svd.components_ + medias_usuario.values[:, None]
    return np.clip(reconstruida, 1, 5), svd

pred_svd, svd = factorizar(entren, n_factores=3)
print(f"RMSE factorización (3 factores): {rmse(pred_svd[filas[es_prueba], cols[es_prueba]]):.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

El RMSE baja de ≈**1.18** (popularidad) a ≈**1.06**: alrededor de un **10 %** menos de error. Parece poco, pero el famoso premio Netflix de 1 millón de dólares se otorgó justamente por mejorar un 10 % el sistema que Netflix ya tenía. La popularidad le da a todos la misma predicción; la factorización **personaliza**.
</details>

### ✋ Ejercicio 6 — ¿Qué descubrió?
Cada factor oculto es una "receta" de películas, como las cargas de PCA. Mira qué películas pesan más (positiva y negativamente) en cada factor. ¿Les puedes poner nombre?

In [ ]:
for i, factor in enumerate(svd.components_):
    orden = np.argsort(factor)
    print(f"Factor {i + 1}:  (+) {[nombres[j] for j in orden[-4:][::-1]]}")
    print(f"           (−) {[nombres[j] for j in orden[:4]]}\n")

✍️ *Mis nombres para los factores:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **Factor 1:** Acción (+) vs. Romance (−): ¡el clásico "¿acción o romance?"!
- **Factor 2:** Animada (+) vs. una mezcla de romance y acción (−): "¿te gustan las animadas?".
- **Factor 3:** una mezcla difícil de nombrar. No todos los factores tienen una interpretación limpia; algunos capturan matices o ruido.

(Los signos son arbitrarios: "Acción (+) vs. Romance (−)" y "Romance (+) vs. Acción (−)" significan lo mismo.)

El modelo **nunca vio** los nombres ni los géneros: solo números de calificaciones. Descubrió los géneros porque explican cómo califica la gente. En la vida real los factores pueden capturar cosas que nadie etiquetaría: "películas de culto", "para ver en familia", "drama intenso"…
</details>

---
## 5. 📖 Hiperparámetros: ¿cuántos gustos ocultos?

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `n_components` (factores latentes) | Cuántos "gustos ocultos" describe el modelo | Más matices, pero empieza a **memorizar** ruido (sobreajuste) | Demasiado simple (subajuste) | 10–100 en sistemas reales; elegir con validación |
| `k` vecinos (filtrado colaborativo) | Cuántos usuarios/productos parecidos se consultan | Predicciones más estables pero menos personales | Más personales pero ruidosas | 20–50 |
| Métrica de similitud | Cómo se mide "parecido" | — | — | Coseno sobre datos **centrados** |
| Mínimo de calificaciones en común | Cuánta evidencia se exige para confiar en una similitud | Similitudes más fiables, menos vecinos | Vecinos "parecidos" por casualidad | 3–5 |
| Regularización (en factorización avanzada) | Castigo a factores extremos (Lección 7.1) | Menos sobreajuste | Más sobreajuste | Ajustar con validación |

### 🔮 Ejercicio 7
Probaremos con 1, 2, 3, 5, 10 y 20 factores. ¿Cómo será la curva de error de prueba? ¿Siempre baja?

✍️ *Mi apuesta:*

In [ ]:
FACTORES = [1, 2, 3, 5, 10, 20]      # ✏️
errores = [rmse(factorizar(entren, n)[0][filas[es_prueba], cols[es_prueba]]) for n in FACTORES]
for n, e in zip(FACTORES, errores):
    print(f"{n:>2} factores → RMSE = {e:.3f}")
plt.plot(FACTORES, errores, "o-"); plt.axhline(rmse(promedio_peli[cols[es_prueba]]), color="gray", ls="--", label="popularidad")
plt.xlabel("Número de factores"); plt.ylabel("RMSE en prueba"); plt.legend(); plt.grid(alpha=.3); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

La curva tiene forma de **U** (Lección 4.4): el error baja hasta **3 factores** (el número real de géneros) y luego **sube** otra vez. Con 20 factores el modelo se pone a "explicar" el ruido de cada calificación y generaliza peor. Es el mismo sobreajuste de siempre, con otra perilla.
</details>

---
## 6. 📖 ¿Qué pasa si…? Los problemas típicos

### ❄️ Arranque en frío (*cold start*)
Llega un **usuario nuevo** sin ninguna calificación. ¿A quién se parece? A nadie: no hay datos. Lo mismo con una **película nueva** que nadie ha visto.

### 🔮 Ejercicio 8
¿Qué le recomendarías a un usuario recién registrado? ¿Y cómo le darías una oportunidad a una película recién estrenada que nadie ha calificado?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **Usuario nuevo:** empieza con **popularidad** (lo más visto o mejor calificado), pregúntale unos pocos gustos al registrarse ("elige 3 géneros que te gusten") o usa datos de contexto (país, dispositivo). En cuanto califique algunas cosas, pasa al filtrado colaborativo.
- **Película nueva:** usa un recomendador **basado en contenido** (género, actores, director: "parecida a otras que te gustaron") o muéstrala a un grupo pequeño para juntar sus primeras calificaciones.

Por eso los sistemas reales son **híbridos**: cada enfoque cubre las debilidades del otro.
</details>

### 🧲 La trampa de "no visto = no me gustó"
Si rellenas las celdas vacías con **0 estrellas** (en vez de "neutral"), el modelo creerá que todos odian todo lo que no han visto. ¡Y la mayoría de las celdas están vacías!

### 🔮 Ejercicio 9
Repetimos la factorización con 3 factores, pero rellenando los huecos con 0 **sin centrar** (el error de principiante). ¿Qué pasará con el RMSE?

✍️ *Mi apuesta:*

In [ ]:
svd_mal = TruncatedSVD(n_components=3, random_state=0)
pred_mal = svd_mal.fit_transform(entren.fillna(0)) @ svd_mal.components_     # 0 = "lo odio" 😬
pred_mal = np.clip(pred_mal, 1, 5)
print(f"RMSE con huecos = 0 (sin centrar): {rmse(pred_mal[filas[es_prueba], cols[es_prueba]]):.2f}")
print(f"RMSE centrando (correcto)        : {rmse(pred_svd[filas[es_prueba], cols[es_prueba]]):.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

El error se dispara a ≈**2.4**, ¡mucho peor que la simple popularidad (≈1.18)! Como el 75 % de las celdas valen 0, el modelo aprende que "lo normal es calificar casi 0" y predice calificaciones bajísimas para todo (que luego recortamos a 1).

Centrar primero hace que el 0 signifique "lo esperado para este usuario", que es una suposición mucho más razonable para lo que no ha visto.
</details>

### 📖 Otros problemas de los recomendadores reales

| 🚨 Problema | Qué pasa | Qué probar |
|---|---|---|
| **Sesgo de popularidad** | Lo popular se recomienda más → se vuelve más popular → nadie descubre lo nuevo | Mezclar con recomendaciones de "descubrimiento"; penalizar lo muy popular |
| **Burbuja de filtro** | Solo te muestra más de lo mismo | Agregar diversidad y algo de azar (exploración vs. explotación, Lección 14.2) |
| **Datos implícitos** | Casi nunca hay estrellas: solo clics, compras o minutos vistos | Tratar como "le interesó / no sabemos", no como "le gustó / no le gustó" |
| **Recomendar lo que ya tiene** | Te recomiendan la lavadora que acabas de comprar | Filtrar lo ya consumido (salvo productos que se recompran) |
| **Evaluar mal** | Medir el error sobre las mismas calificaciones de entrenamiento | Esconder calificaciones (como aquí) o mejor, pruebas A/B con usuarios reales |
| **Usuarios con pocas calificaciones** | Similitudes "perfectas" por casualidad con 1–2 películas en común | Exigir un mínimo de calificaciones en común |

---
## 7. 📋 Ficha del modelo

| | Filtrado colaborativo (vecinos) | Factorización de matrices |
|---|---|---|
| **Tipo** | No supervisado (descubre patrones de gusto) | Igual |
| **Intuición** | El dependiente del videoclub: gente como tú también disfrutó… | Cada usuario y cada producto se describen con unos pocos "gustos ocultos" |
| **Cómo funciona** | Similitud coseno entre usuarios (o productos) + promedio ponderado de los vecinos | Descompone la matriz (p. ej. `TruncatedSVD`) en factores de usuarios × factores de productos |
| **Hiperparámetros clave** | `k` vecinos, métrica, mínimo en común | Número de factores, regularización |
| **✅ Fortalezas** | Muy explicable ("porque te gustó X") · No necesita saber nada del producto | Escala a millones · Suele ser más preciso · Factores interpretables a veces |
| **❌ Debilidades** | Lento con muchos usuarios · Sufre con matrices muy vacías | Menos explicable · Hay que elegir el número de factores |
| **Ambos sufren…** | Arranque en frío, sesgo de popularidad, burbuja de filtro | |
| **Úsalo cuando…** | Catálogo mediano, necesitas explicar recomendaciones | Catálogo y usuarios grandes, buscas precisión |

## 8. 🧠 Resumen en 5 puntos
1. Un recomendador parte de una **matriz usuarios × productos** casi vacía y adivina las celdas que faltan.
2. **Filtrado colaborativo:** usa la **similitud coseno** (sobre datos **centrados**) para encontrar usuarios o productos parecidos.
3. **Factorización de matrices:** descubre **gustos ocultos** (como PCA) que explican las calificaciones; el número de factores se elige con validación (curva en U).
4. Siempre compáralo contra la **popularidad** y evalúa con calificaciones **escondidas** (o pruebas A/B).
5. Problemas típicos: **arranque en frío**, **sesgo de popularidad**, **burbuja de filtro** y tratar "no visto" como "no me gustó". Los sistemas reales son **híbridos**.

---
## 9. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 9.1 Repaso rápido (opción múltiple)

**1.** La similitud coseno entre dos usuarios vale −0.9. ¿Qué significa?
- a) Gustos casi idénticos · b) Gustos casi opuestos · c) No tienen nada que ver · d) Error de cálculo

**2.** "Quienes compraron este producto también compraron…" es un ejemplo de…
- a) Popularidad · b) Filtrado colaborativo basado en productos · c) Basado en contenido · d) Regresión

**3.** Un usuario recién registrado, sin ninguna calificación. ¿Qué enfoque **no** puede ayudarle todavía?
- a) Popularidad · b) Preguntarle sus géneros favoritos · c) Filtrado colaborativo basado en usuarios · d) Contenido

**4.** Subes el número de factores de 5 a 200 y el RMSE de **prueba** empeora. ¿Qué pasó?
- a) Subajuste · b) Sobreajuste · c) Fuga de datos · d) Arranque en frío

**5.** ¿Por qué centramos las calificaciones de cada usuario antes de compararlos?
- a) Para que corra más rápido · b) Para que los usuarios "generosos" y "exigentes" se puedan comparar y un hueco signifique "neutral" · c) Porque la similitud coseno no acepta números negativos · d) No hace falta

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Las flechas apuntan en direcciones opuestas.
2. **b)** Compara productos según la gente que los compra.
3. **c)** Sin calificaciones no hay forma de saber a quién se parece.
4. **b)** Demasiados factores: memoriza el ruido.
5. **b)** Y la c) es falsa: el coseno acepta negativos sin problema.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Una plataforma de cursos en línea solo tiene datos de **qué cursos empezó** cada persona (no calificaciones). ¿Qué problemas trae eso? ¿Qué otras señales usarías para saber si un curso le gustó a alguien?

**B. Caso de análisis 💼** Un supermercado en línea pone su recomendador en marcha y las ventas totales **no** suben, aunque el RMSE de prueba era excelente. Al revisar, el 80 % de las recomendaciones son leche, huevos y pan. ¿Qué está pasando y qué propondrías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Son **datos implícitos**: empezar un curso no significa que te gustó (quizá lo abandonaste en 5 minutos), y no empezarlo no significa que no te guste (quizá nunca lo viste). Señales útiles: **porcentaje completado**, tiempo dedicado, si volvió a entrar, si hizo los ejercicios, si se inscribió en otro curso del mismo autor. Combinarlas en una "calificación implícita" (p. ej. 1 si completó más del 50 %).

**B.** El recomendador predice muy bien lo que la gente **ya iba a comprar de todos modos** (sesgo de popularidad + productos de recompra): no aporta nada. Un RMSE bajo no es lo mismo que **valor para el negocio**. Propuestas: excluir productos básicos que se recompran solos; medir el éxito con una **prueba A/B** de ventas **adicionales** (no con RMSE); favorecer recomendaciones de **descubrimiento** (productos que el cliente no suele comprar pero que compran sus "gemelos").
</details>

### 9.3 🧩 Práctica integradora: "Quienes compraron esto también compraron…"

Una tienda en línea de artículos para el hogar tiene el historial de **compras** de 500 clientes en 30 productos (1 = lo compró, 0 = no). No hay estrellas: solo compras. Quiere una sección "**Quienes compraron esto también compraron…**" y recomendaciones personalizadas.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Los datos.** Los productos pertenecen, sin que el modelo lo sepa, a 3 "mundos": cocina, jardín y mascotas.

In [ ]:
rng = np.random.default_rng(7)
mundos = {"cocina": ["sartén", "cuchillos", "licuadora", "tabla_picar", "cafetera", "especiero",
                     "olla_presión", "batidora", "moldes", "delantal"],
          "jardín": ["manguera", "tijeras_podar", "macetas", "tierra", "semillas", "regadera",
                     "guantes_jardín", "fertilizante", "pala", "aspersor"],
          "mascotas": ["croquetas", "correa", "cama_perro", "juguete_gato", "arenero", "plato_mascota",
                       "cepillo_pelo", "transportadora", "collar", "snacks_mascota"]}
productos = sum(mundos.values(), [])
mundo_de = np.repeat(np.arange(3), 10)
interes = rng.random((500, 3)) ** 3                     # cada cliente se interesa sobre todo por 1 o 2 mundos
prob_compra = 0.03 + 0.6 * interes[:, mundo_de] * rng.uniform(0.5, 1.5, 30)
compras = pd.DataFrame((rng.random((500, 30)) < prob_compra).astype(int), columns=productos)
print(f"Clientes × productos: {compras.shape} · compras totales: {compras.values.sum()}")
print("\nLos 5 productos más vendidos:")
print(compras.sum().sort_values(ascending=False).head())

**Paso 2 🔮 — "También compraron".** Calcula la similitud coseno entre **productos** (columnas). Con datos de compras (0/1) no hace falta centrar: 0 significa "no lo compró". ¿Qué productos crees que saldrán como los más parecidos a la **sartén**?

✍️ *Mi apuesta:*

In [ ]:
sim_prod = pd.DataFrame(similitud_coseno(compras.T.to_numpy().astype(float)), index=productos, columns=productos)

PRODUCTO = "sartén"     # ✏️ prueba "manguera" o "croquetas"
print(f"Quienes compraron '{PRODUCTO}' también compraron:")
print(sim_prod[PRODUCTO].drop(PRODUCTO).sort_values(ascending=False).head(5).round(2))

<details><summary><b>👀 Ver respuesta</b></summary>

Los 5 más parecidos a la sartén son **todos productos de cocina** (licuadora, especiero, cuchillos, etc.). Con "manguera" salen productos de jardín y con "croquetas", de mascotas. El sistema descubrió los 3 "mundos" sin que nadie se los dijera, solo mirando qué se compra junto.
</details>

**Paso 3 ▶️ — Recomendaciones personalizadas.** Para cada cliente sumamos la similitud de cada producto con **todo lo que ya compró**, y recomendamos los 5 con mayor puntaje que **todavía no tiene**.

In [ ]:
def recomendar(cliente, n=5):
    tiene = compras.loc[cliente]
    puntaje = sim_prod.loc[:, tiene == 1].sum(axis=1)       # parecido con todo lo que ya compró
    puntaje = puntaje[tiene == 0]                            # no recomendar lo que ya tiene
    return list(puntaje.sort_values(ascending=False).head(n).index)

CLIENTE = 3     # ✏️ prueba otros clientes
print("Ya compró     :", list(compras.columns[compras.loc[CLIENTE] == 1]))
print("Le recomiendo :", recomendar(CLIENTE))

✋ ¿Tienen sentido las recomendaciones para este cliente? ¿De qué "mundo" son?

<details><summary><b>👀 Ver respuesta</b></summary>

El cliente 3 compró unas tijeras de podar y una pala: es del mundo **jardín**. Las 5 recomendaciones (fertilizante, macetas, guantes, regadera, tierra) son **todas de jardín** y no las tiene todavía. Prueba otros clientes: las recomendaciones siguen el "mundo" de cada uno.
</details>

**Paso 4 🔮 — ¿Funciona mejor que la popularidad?** Evaluación honesta: para cada cliente con al menos 3 compras, **escondemos una** compra al azar, recomendamos 5 productos con el resto y miramos si el producto escondido está entre los 5 (*tasa de acierto en 5*, *hit rate@5*). Lo comparamos con recomendar siempre **los 5 más vendidos** (que el cliente no tenga). ¿Qué tasa esperas para cada uno?

✍️ *Mi apuesta:*

In [ ]:
rng_eval = np.random.default_rng(0)
aciertos_cf, aciertos_pop, evaluados = 0, 0, 0
popularidad = compras.sum()

for c in compras.index:
    compradas = np.where(compras.loc[c] == 1)[0]
    if len(compradas) < 3:
        continue
    escondido = productos[rng_eval.choice(compradas)]
    sin_escondido = compras.copy()
    sin_escondido.loc[c, escondido] = 0                                   # lo escondemos
    tiene = sin_escondido.loc[c]
    sim = pd.DataFrame(similitud_coseno(sin_escondido.T.to_numpy().astype(float)), index=productos, columns=productos)
    puntaje = sim.loc[:, tiene == 1].sum(axis=1)[tiene == 0]
    aciertos_cf += escondido in puntaje.nlargest(5).index
    aciertos_pop += escondido in popularidad[tiene == 0].nlargest(5).index
    evaluados += 1

print(f"Clientes evaluados: {evaluados}")
print(f"Hit rate@5 — filtrado colaborativo: {aciertos_cf / evaluados:.0%}")
print(f"Hit rate@5 — los más vendidos      : {aciertos_pop / evaluados:.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

El filtrado colaborativo acierta en ≈**50 %** de los clientes (¡la mitad de las veces el producto escondido está entre sus 5 recomendaciones!), contra ≈**33 %** de los más vendidos. Como referencia, recomendar 5 productos al azar entre los ~24 que el cliente no tiene acertaría ≈20 %. La popularidad mejora algo al azar, pero mezcla "mundos": lo más vendido de jardín no le sirve a quien solo compra cosas de cocina.

> 📝 Recalculamos las similitudes **sin** la compra escondida para no hacer trampa (fuga de datos, Lección 2.6). Es lento a propósito: así se ve que cada evaluación es honesta.
</details>

**Paso 5 ✋ — El informe.** En dos o tres frases para la gerente de la tienda: ¿vale la pena el recomendador? ¿Qué limitaciones tiene?

✍️ *Mi informe:*

<details><summary><b>👀 Ver respuesta</b></summary>

*"El recomendador 'quienes compraron esto también compraron' acierta el siguiente producto de un cliente la mitad de las veces, frente a un tercio si mostramos los más vendidos (≈50 % vs. ≈33 % en una prueba con compras escondidas). Funciona porque descubre solo qué productos se compran juntos (cocina, jardín, mascotas). Limitaciones: no sirve para clientes nuevos ni para productos recién agregados (para ellos usaremos los más vendidos de cada categoría), y antes de ampliarlo conviene confirmar con una prueba A/B que de verdad aumenta las ventas."*
</details>

---
## 📚 Para profundizar
- 📘 **Libro:** *Mining of Massive Datasets* (Leskovec, Rajaraman y Ullman), capítulo 9 "Recommendation Systems": gratuito en línea.
- 📺 **Video:** Google for Developers, curso *Recommendation Systems* (inglés, con subtítulos).
- 🛠️ **Herramientas:** `surprise` (para Colab: `!pip install scikit-surprise`) implementa filtrado colaborativo y factorización con un estilo parecido a scikit-learn; `implicit` para datos de compras/clics.

---
⬅️ **Anterior:** [8.6 Detección de anomalías: Isolation Forest y compañía](8.6-deteccion-de-anomalias.ipynb) · ➡️ **Siguiente:** [8.8 Proyecto del módulo: Los clientes del distribuidor 🚚](8.8-proyecto-del-modulo.md)